# Happiness in the music we listen to, and how we respond to it
### By Quinn Epstein
---
## Setup for graphing

In [1]:
from re import template

import opendatasets as od
import pandas as pd
import plotly.graph_objects as go
from fontTools.cffLib import width
from fontTools.ttLib.tables import C_O_L_R_
from matplotlib.pyplot import title
from plotly.graph_objs.scatter3d import line

# too large for GitHub upload
od.download('https://www.kaggle.com/datasets/ektanegi/spotifydata-19212020')
od.download('https://www.kaggle.com/datasets/mathurinache/world-happiness-report')

spotifyStats = pd.read_csv('spotifydata-19212020/data.csv')

# World happiness comes separate per year, needs to be stitched together
years = [2015, 2016, 2017, 2018, 2019, 2020] #years in happiness set
frames = [];
for year in years:
    currYear = pd.read_csv(f'world-happiness-report/{year}.csv');
    #fix diff names for columns
    if 'Happiness Score' in currYear.columns:
        scoreCol = 'Happiness Score';
        countryCol = 'Country';
    elif 'Happiness.Score' in currYear.columns:
        scoreCol = 'Happiness.Score';
        countryCol = 'Country';
    elif 'Score' in currYear.columns:
        scoreCol = 'Score';
        countryCol = 'Country or region';
    elif 'Ladder score' in currYear.columns:
        scoreCol = 'Ladder score';
        countryCol = 'Country name';

    #build new df
    temp = currYear[[countryCol, scoreCol]].copy();
    temp.rename(columns={countryCol: 'Country', scoreCol: 'Happiness Score'}, inplace=True);
    temp['Year'] = year;
    frames.append(temp);

happinessScores : pd.DataFrame = pd.concat(frames, ignore_index=True);

# Level scores to same base
happinessScores['NormalizedScores'] = happinessScores['Happiness Score'] / 10;

Skipping, found downloaded files in ".\spotifydata-19212020" (use force=True to force download)
Skipping, found downloaded files in ".\world-happiness-report" (use force=True to force download)


---
## First im going to make two graphs showing happiness overtime 

In [23]:
# the music dataset uses valence for happiness
yearlyValence = spotifyStats.groupby('year')['valence'].mean().reset_index();
yearlyHappiness = happinessScores.groupby('Year')['NormalizedScores'].mean().reset_index();

musicYears = go.Figure();
musicYears.add_trace(go.Scatter(
    x = yearlyValence['year'],
    y = yearlyValence['valence'],
    mode = 'lines+markers',
    name = 'Music Happiness (Valence)',
    line = dict(color = 'mediumpurple', width = 3)
));

musicYears.update_layout(
    title = 'Decline in Music Happiness Over 100 Years',
    xaxis_title = 'Year',
    yaxis_title = 'Music Positivity (0.0 - 1.0)',
    template = 'simple_white+plotly_dark'
);

musicYears.show();